In [ ]:
from src.generation import (
    three_d_perlin,
    split_points,
    cull_below_threshold,
    cull_within_radius,
    coordinate_to_vector,
    icosahedron,
    subdivide_polygon,
    vector_to_coordinate,
    normalize_vector,
    dual_polygon,
    march_ray_3d,
    translate_data,
    apply_dataseries_to_polygon,
)
from src.data import DataSeries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection


In [ ]:
grid_size = [5] * 3
centering_translation = np.array([-size / 2 for size in grid_size], dtype=float)
sampling_translation = np.array([4.0, -5.0, 0.0], dtype=float)
arr = (
    DataSeries.three_d_perlin(
        *grid_size,
        seed=42,
        resolution=0.20,
        scale=4.0,
        translation=sampling_translation,
        repeat=(255, 255, 255),
    )
    .translate_data(centering_translation - sampling_translation)
    .cull_below_threshold(threshold=0.0)
    .cull_within_radius(radius=2, center_point=(0.0, 0.0, 0.0))
)

# only now split for plotting
coords, values = split_points(arr.data)

x, y, z = coords.T
for rot_deg in range(0, 360, 45):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    sc = ax.scatter(x, y, z, c=values, cmap="plasma", s=20)
    ax.view_init(elev=30, azim=rot_deg)
    plt.colorbar(sc, ax=ax, label="Perlin value")
    plt.show()

In [ ]:
# unit vectors from arr with coordinate_to_vector
# coordinate_to_vector only accepts one coordinate at a time,
# do no use raw python loops
# arr shape is [[z,y,z],value]
# unit_vectors = np.array([coordinate_to_vector(coord) for coord in arr[:, 0]])
# coords, values = split_points(unit_vectors)
# x, y, z = coords.T
# for rot_deg in range(0, 360, 45):
#      fig = plt.figure()
#      ax = fig.add_subplot(projection="3d")
#      sc = ax.scatter(x, y, z, c=values, cmap="plasma", s=20)
#      ax.view_init(elev=30, azim=rot_deg)
#      plt.colorbar(sc, ax=ax, label="Perlin value")
#      plt.show()

In [ ]:
# ico = icosahedron(radius=1.0)
# vectors = np.stack(ico[0])

#

# vec_to_coord = np.vectorize(lambda d, m: vector_to_coordinate([d, m]), otypes=[np.ndarray])
# coords = np.vstack(vec_to_coord(vectors[:, 0], vectors[:, 1]))

# values = vectors[:, 1]
# print(coords.shape, values.shape)

# for rot_deg in range(0, 360, 45):
#      fig = plt.figure()
#      ax = fig.add_subplot(projection="3d")
#      sc = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=values, cmap="plasma", s=20)
#      ax.view_init(elev=30, azim=rot_deg)
#      plt.colorbar(sc, ax=ax, label="Perlin value")
#      plt.show()

In [ ]:
ico = icosahedron(radius=1.0)
ico = dual_polygon(subdivide_polygon(ico, subdivisions=3))
vectors, edges, faces = ico
# convert to ndarray form3)
vectors = np.stack(vectors)

# --- normalize all vectors in bulk ---
directions = np.stack(vectors[:, 0]).astype(float)
magnitudes = vectors[:, 1].astype(float)

norms = np.linalg.norm(directions, axis=1, keepdims=True)
safe_norms = np.where(norms == 0, 1.0, norms)
normalized_dirs = directions / safe_norms
normalized_mags = np.ones_like(magnitudes, dtype=float)

vectors = np.empty((len(normalized_dirs), 2), dtype=object)
vectors[:, 0] = [d.astype(np.float32) for d in normalized_dirs]
vectors[:, 1] = normalized_mags

# --- convert to coordinates in bulk ---
coords = normalized_dirs * normalized_mags[:, None]
values = normalized_mags

# --- plotting ---
for rot_deg in range(0, 360, 90):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    # scatter vertices
    sc = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                    c=values, cmap="plasma", s=50, depthshade=True)

    # edges
    edge_lines = np.stack([coords[edges[:, 0]], coords[edges[:, 1]]], axis=1)
    ax.add_collection3d(Line3DCollection(edge_lines, colors="black", linewidths=1))

    # faces
    face_polys = [coords[face] for face in faces]
    ax.add_collection3d(Poly3DCollection(face_polys,
                                         facecolors="cyan", edgecolors="k",
                                         linewidths=0.5, alpha=0.2))

    ax.view_init(elev=30, azim=rot_deg)
    ax.set_box_aspect([1, 1, 1])

    plt.colorbar(sc, ax=ax, label="Magnitude")
    plt.show()


In [ ]:
vectors, edges, faces = apply_dataseries_to_polygon(
    dataseries=arr,
    polygon=ico,
    default=-10.0,
    ray_radius=0.50,
    ray_step=0.25,
    max_march=2.5
)
print("completed applying data series on polygon...")
r = 0.1
scaled_vectors = np.array(
    [np.array([v[0], r, v[2]], dtype=object) for v in vectors],
    dtype=object,
)
print("completed scaling vectors...")

# convert to coordinates using helper
coords = np.stack([vector_to_coordinate(v[:2], origin_point=(0,0,0)) for v in scaled_vectors])
values = np.array([v[2] for v in scaled_vectors], dtype=float)
print("completed converting to coordinates...")

# build face geometry and average colors
face_coords = [coords[f] for f in faces]
face_colors = np.array([values[f].mean() for f in faces])
print("completed building face geometry and colors...")

print("Start plotting...")
for rot_deg in range(0, 360, 90):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    # edges
    edge_lines = np.stack([coords[edges[:, 0]], coords[edges[:, 1]]], axis=1)
    ax.add_collection3d(Line3DCollection(edge_lines, colors="black", linewidths=1))

    # faces colored by average value
    ax.add_collection3d(Poly3DCollection(
        face_coords,
        facecolors=plt.cm.plasma(face_colors),
        edgecolors="k",
        linewidths=0.5,
        alpha=1.0,
    ))

    ax.view_init(elev=30, azim=rot_deg)
    ax.set_box_aspect([1, 1, 1])

    mappable = plt.cm.ScalarMappable(cmap="plasma")
    mappable.set_array(face_colors)
    plt.colorbar(mappable, ax=ax, label="Face mean value")

    plt.show()
